# External liver-mask adapter

**Scientific purpose.** Map an available external liver mask to CT geometry and record
whether the downstream isotropic adapter must use its documented tumor-dilation fallback.
external adapter.

**Runnability classification:** runnable after notebook 10 with private NIfTI inputs.
**Inputs:** retained external CT/tumor masks and any reviewed liver mask.
**Private outputs:** a patient-linked liver-adapter manifest and derived masks.

This stage does not fit, reconstruct, or save W5 models. Exact serialized historical W5
model objects were not retained as public artifacts.


In [ ]:
from pathlib import Path
import sys


def locate_repository(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from within the cloned repository.")


REPO_ROOT = locate_repository()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from liver_tumor_pipeline.config import load_config, require_path

CONFIG_PATH = REPO_ROOT / "configs" / "paths.yaml"
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(
        "Create an untracked configs/paths.yaml from configs/paths.example.yaml and "
        "set the documented environment variables before running this workflow."
    )

CONFIG = load_config(CONFIG_PATH)
PRIVATE_ARTIFACT_ROOT = require_path(CONFIG, "private_artifact_root", must_exist=False)
OUTPUT_ROOT = require_path(CONFIG, "output_root", must_exist=False)

import numpy as np
import pandas as pd
import SimpleITK as sitk

EXTERNAL_ROOT = PRIVATE_ARTIFACT_ROOT / "external"
FILTERED_MANIFEST_PATH = EXTERNAL_ROOT / "tumor_masks" / "external_manifest_filtered.csv"
ADAPTER_ROOT = EXTERNAL_ROOT / "liver_adapter"
ADAPTER_MASK_ROOT = ADAPTER_ROOT / "masks"
ADAPTER_MANIFEST_PATH = ADAPTER_ROOT / "external_liver_adapter_manifest.csv"
if not FILTERED_MANIFEST_PATH.is_file():
    raise FileNotFoundError("Filtered private external manifest is unavailable; run notebook 10.")
ADAPTER_MASK_ROOT.mkdir(parents=True, exist_ok=True)


## Geometry adapter

A provided liver segmentation is resampled to the reference CT with nearest-neighbor
interpolation. If no usable liver mask exists, this stage records the absence; notebook 12
then dilates the tumor mask by 15 voxels after 1-mm resampling. That fallback differs from the internal
expert-liver-mask pipeline and is an acknowledged source of domain shift.


In [ ]:
def read_binary(path: Path) -> tuple[sitk.Image, np.ndarray]:
    image = sitk.ReadImage(str(path))
    return image, (sitk.GetArrayFromImage(image) > 0).astype(np.uint8)


def adapt_liver_mask(
    ct_path: Path,
    tumor_mask_path: Path,
    output_path: Path,
    liver_mask_path: Path | None = None,
) -> dict:
    ct_image = sitk.ReadImage(str(ct_path))
    tumor_image, tumor = read_binary(tumor_mask_path)
    tumor_on_ct = sitk.Resample(
        tumor_image,
        ct_image,
        sitk.Transform(),
        sitk.sitkNearestNeighbor,
        0,
        sitk.sitkUInt8,
    )
    tumor = (sitk.GetArrayFromImage(tumor_on_ct) > 0).astype(np.uint8)
    if tumor.sum() == 0:
        raise ValueError("External tumor mask is empty after CT-geometry mapping.")

    used_segmented_liver = liver_mask_path is not None and liver_mask_path.is_file()
    if used_segmented_liver:
        liver_image = sitk.ReadImage(str(liver_mask_path))
        liver_on_ct = sitk.Resample(
            liver_image,
            ct_image,
            sitk.Transform(),
            sitk.sitkNearestNeighbor,
            0,
            sitk.sitkUInt8,
        )
        liver = (sitk.GetArrayFromImage(liver_on_ct) > 0).astype(np.uint8)
        if liver.sum() == 0:
            raise ValueError("Provided liver mask is empty after CT-geometry mapping.")
        output = sitk.GetImageFromArray(liver)
        output.CopyInformation(ct_image)
        sitk.WriteImage(output, str(output_path), useCompression=True)
        return {
            "liver_mask_path": str(output_path),
            "liver_mask_strategy": "segmented_liver",
            "liver_voxels": int(liver.sum()),
        }
    return {
        "liver_mask_path": "",
        "liver_mask_strategy": "defer_tumor_dilation_until_1mm_resampling",
        "liver_voxels": 0,
    }


## Private manifest construction

For each retained row, choose the available C2/primary CT and its mapped tumor mask,
optionally supply a reviewed liver segmentation, and call `adapt_liver_mask`. Join the
returned fields to the private filtered manifest and save it to
`ADAPTER_MANIFEST_PATH`. Do not expose patient-linked paths or adapter decisions.

The external adapter's resampling, crop construction, phase availability, segmentation,
and missing-feature handling differ from the internal workflow. External degradation
cannot be attributed to a single cause.
